In [1]:
import altair as alt
import gcsfs
import pandas as pd


GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [2]:
GCS = "gs://calitp-analytics-data/data-analyses/ntd/"
orig_df = pd.read_parquet(
    f"{GCS}raw_transit_performance_metrics_data.parquet",
    filesystem = gcsfs.GCSFileSystem()
)
orig_df.dtypes

agency_name           object
agency_status         object
city                  object
mode                  object
service               object
ntd_id                object
reporter_type         object
reporting_module      object
state                 object
primary_uza_name      object
year                  object
upt                    int64
vrh                    int64
vrm                    int64
opexp_total            int64
RTPA                  object
_merge              category
dtype: object

In [3]:
orig_df.head(2)

,agency_name,agency_status,city,mode,service,ntd_id,reporter_type,reporting_module,state,primary_uza_name,year,upt,vrh,vrm,opexp_total,RTPA,_merge
0,City of Porterville (COLT) - Transit Department,Active,Porterville,Demand Response,Purchased Transportation,90198,Building Reporter,Urban,CA,"Porterville, CA",2019,13112,2997,43696,572799,Tulare County Association of Governments,both
1,City of Porterville (COLT) - Transit Department,Active,Porterville,Demand Response,Purchased Transportation,90198,Building Reporter,Urban,CA,"Porterville, CA",2020,11523,3669,48138,686165,Tulare County Association of Governments,both


# what columns are needed
* RTPA - shows `rtpa_name`, not `rtpa_name_split`
* just read in the subset of columns needed, these metrics are precalculated

In [4]:
crosswalk = pd.read_parquet(
    f"{GCS_FILE_PATH}crosswalk2.parquet", 
    filesystem=gcsfs.GCSFileSystem(),
    columns = ["ntd_id_2022", "rtpa_name"]
).rename(columns = {"ntd_id_2022": "ntd_id"})

df = pd.read_parquet(
    f"{GCS_FILE_PATH}annual.parquet",
    # should only certain columns be read in? now this table is much larger
    filesystem=gcsfs.GCSFileSystem(),
    columns = [
        "source_agency", "agency_status", "source_city", 
        "mode", "type_of_service", "ntd_id", 
        "reporter_type", "reporting_module", "source_state", "primary_uza_name",
        "year", "unlinked_passenger_trips", "vehicle_revenue_hours", "vehicle_revenue_miles",
        "operating_expenses_total",
    ]
).merge(
    crosswalk,
    on = "ntd_id",
    how = "left"
)

# mode should be mode_full_name
# service refers to type_of_service_full name 
df.dtypes

source_agency               object
agency_status               object
source_city                 object
mode                        object
type_of_service             object
ntd_id                      object
reporter_type               object
reporting_module            object
source_state                object
primary_uza_name            object
year                         Int64
unlinked_passenger_trips     Int64
vehicle_revenue_hours        Int64
vehicle_revenue_miles        Int64
operating_expenses_total     Int64
rtpa_name                   object
dtype: object

In [5]:
df2 = df[df.year <= 2023].reset_index(drop=True)

In [6]:
orig_df.shape, df2.shape

((2091, 17), (2676, 16))

In [7]:
# there are some additional ones in the dbt model
m1 = pd.merge(
    orig_df[["ntd_id"]].drop_duplicates(),
    df2[["ntd_id"]].drop_duplicates(),
    on = ["ntd_id", ],
    how = "outer",
    indicator=True
)
    
m1._merge.value_counts()

_merge
both          168
right_only     15
left_only       0
Name: count, dtype: int64

In [8]:
m1[m1._merge == "right_only"]

,ntd_id,_merge
0,30109,right_only
1,30131,right_only
99,90235,right_only
101,90238,right_only
106,90249,right_only
164,90313,right_only
165,90314,right_only
170,91092,right_only
171,99256,right_only
172,99262,right_only


These still get aggregated by agency/mode/tos.

In [9]:
import B3_ntd_utils as ntd_utils

val_cols = [
    "opex_per_vrh",
    "opex_per_vrm",
    "upt_per_vrh",
    "upt_per_vrm",
    "opex_per_upt",
]

label_dict ={
    'unlinked_passenger_trips':"Unlinked Passenger Trips",
    'vehicle_revenue_miles':"Vehicle Revenue Miles",
    'vehicle_revenue_hours':"Vehicle Revenue Hours",
    'operating_expenses_total':"Operating Expense Total",
    'opex_per_vrh':"Operating Expense per Vehicle Revenue Hours",
    'opex_per_vrm':"Operating Expense per Vehicle Revenue Miles",
    'opex_per_upt':"Operating Expense per Unlinked Passenger Trips",
    'upt_per_vrh':"Unlinked Passenger Trips per Vehicle Revenue Hours",
    'upt_per_vrm':"Unlinked Passenger Trips per Vehicle Revenue Miles",
}

def make_long(df: pd.DataFrame, group_cols: list, value_cols: list):
    """
    melts dataframes to get all the metrics into a single column for better charting
    do the labeling here too
    this function can sit in notebook
    """
    df_long = df[group_cols + value_cols].melt(
        id_vars=group_cols,
        value_vars=value_cols,
    )

    df_long = df_long.assign(
        label = df_long.variable.map(label_dict)
    )

    return df_long

by_agency = ntd_utils.calculate_efficiency_metrics_by_group(
    df2, ["ntd_id", "source_agency", "rtpa_name", "year"]
).pipe(
    make_long, 
    ["ntd_id", "source_agency", "rtpa_name", "year"], 
    val_cols
)
# agency_name...is this renamed from source_agency?

In [10]:
by_mode = ntd_utils.calculate_efficiency_metrics_by_group(
    df2, ["mode", "rtpa_name", "year"]
).pipe(
    make_long, 
    ["mode", "rtpa_name", "year"], 
    val_cols
)

by_mode

,mode,rtpa_name,year,variable,value,label
0,CB,El Dorado County Transportation Commission,2018,opex_per_vrh,196.13,Operating Expense per Vehicle Revenue Hours
1,CB,El Dorado County Transportation Commission,2019,opex_per_vrh,193.45,Operating Expense per Vehicle Revenue Hours
2,CB,El Dorado County Transportation Commission,2020,opex_per_vrh,188.83,Operating Expense per Vehicle Revenue Hours
3,CB,El Dorado County Transportation Commission,2021,opex_per_vrh,241.0,Operating Expense per Vehicle Revenue Hours
4,CB,El Dorado County Transportation Commission,2022,opex_per_vrh,221.87,Operating Expense per Vehicle Revenue Hours
...,...,...,...,...,...,...
2905,YR,San Diego Association of Governments,2019,opex_per_upt,8.94,Operating Expense per Unlinked Passenger Trips
2906,YR,San Diego Association of Governments,2020,opex_per_upt,11.37,Operating Expense per Unlinked Passenger Trips
2907,YR,San Diego Association of Governments,2021,opex_per_upt,18.29,Operating Expense per Unlinked Passenger Trips
2908,YR,San Diego Association of Governments,2022,opex_per_upt,21.62,Operating Expense per Unlinked Passenger Trips


In [11]:
by_tos = ntd_utils.calculate_efficiency_metrics_by_group(
    df2, ["type_of_service", "rtpa_name", "year"]
).pipe(
    make_long, 
    ["type_of_service", "rtpa_name", "year"], 
    val_cols
)

by_tos

,type_of_service,rtpa_name,year,variable,value,label
0,DO,Del Norte Local Transportation Commission,2018,opex_per_vrh,56.63,Operating Expense per Vehicle Revenue Hours
1,DO,Del Norte Local Transportation Commission,2019,opex_per_vrh,99.19,Operating Expense per Vehicle Revenue Hours
2,DO,Del Norte Local Transportation Commission,2020,opex_per_vrh,168.52,Operating Expense per Vehicle Revenue Hours
3,DO,Del Norte Local Transportation Commission,2021,opex_per_vrh,512.08,Operating Expense per Vehicle Revenue Hours
4,DO,Del Norte Local Transportation Commission,2022,opex_per_vrh,447.41,Operating Expense per Vehicle Revenue Hours
...,...,...,...,...,...,...
1645,TX,Transportation Agency for Monterey County,2019,opex_per_upt,NaN,Operating Expense per Unlinked Passenger Trips
1646,TX,Transportation Agency for Monterey County,2020,opex_per_upt,NaN,Operating Expense per Unlinked Passenger Trips
1647,TX,Transportation Agency for Monterey County,2021,opex_per_upt,NaN,Operating Expense per Unlinked Passenger Trips
1648,TX,Transportation Agency for Monterey County,2022,opex_per_upt,NaN,Operating Expense per Unlinked Passenger Trips


## Chart Ideas

* existing charts put cost-efficiency, service-effectiveness metrics in separate headings, and under those headings, there are by agency, by mode, by TOS
* re-organize this to put cost-efficiency, service-effectiveness metrics side-by-side for agency, then side-by-side for mode, side-by-side for TOS?

In [34]:
test_tos= by_tos[by_tos.rtpa_name.str.contains("Del Norte")]#.variable.value_counts()

test_tos2 = test_tos.sort_values(["rtpa_name", "type_of_service", "variable", "year"]).groupby(
    ["rtpa_name", "type_of_service", "variable"]
).agg({
    "value": lambda x: list(x)
}).reset_index()

In [23]:
from great_tables import GT
import gt_extras as gte
import polars as pl

In [38]:
test_tos3 = pl.from_pandas(test_tos2)
(GT(test_tos3, rowname_col="variable")
    .fmt_nanoplot(columns="value")
)

,rtpa_name,type_of_service,value
opex_per_upt,Del Norte Local Transportation Commission,DO,96717.817.822.276.696710918.0
opex_per_vrh,Del Norte Local Transportation Commission,DO,51256.656.699.2169512447139
opex_per_vrm,Del Norte Local Transportation Commission,DO,28.24.174.176.309.8028.226.810.6
upt_per_vrh,Del Norte Local Transportation Commission,DO,7.680.533.184.472.200.534.127.68
upt_per_vrm,Del Norte Local Transportation Commission,DO,0.590.0300.230.280.130.0300.250.59
